[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_01_cnn.ipynb)

# S11 · 01 — Redes convolucionales (CNN): cómo «ve» una red

Sprint 11 · UD 4.4

**Contexto TurisData.** El cliente tiene miles de fotos de alojamientos, zonas comunes y platos, y hoy nadie las usa. Antes de clasificarlas necesitas entender cómo una red **procesa una imagen**. Aquí lo verás con imágenes pequeñas que se ejecutan sin internet.

**Este cuaderno tiene 4 ejercicios y un mini-reto.** Los ejercicios 1 y 2 son los mismos que los del cuaderno «en clase» (allí, del 1 al 2).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar los dígitos y verlos como matrices

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué forma tiene el conjunto y qué significa cada número?
2. ¿Qué valores puede tener un píxel y qué significan?
3. ¿Qué forma tendría un lote de fotos en color de 96×96?

In [ ]:
digitos = load_digits()
imagenes, etiquetas = digitos.images, digitos.target      # (1797, 8, 8) y (1797,)
print("Forma del conjunto:", imagenes.shape, "· valores de", imagenes.min(), "a", imagenes.max())
print("Un dígito (matriz 8×8):\n", imagenes[0].astype(int))

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.4))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow(imagenes[i], cmap="gray_r"); eje.set_title(f"es un {etiquetas[i]}", fontsize=9); eje.axis("off")
plt.suptitle("Dígitos 8×8 de scikit-learn (miniaturas de 64 píxeles)"); plt.tight_layout(); plt.show()

### ✏️ A3 · EJERCICIO 1 de 4 — Preparar los datos para una red

**Tu turno:** Prepara los datos para una red: en `X` guarda las imágenes **normalizadas entre 0 y 1** (divide entre 16) con forma `(n, 8, 8, 1)` (Keras espera un canal explícito). Después crea con `train_test_split` (`test_size=0.25`, `random_state=SEMILLA`, `stratify=etiquetas`) los conjuntos `X_ent, X_pru, y_ent, y_pru`.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Por qué se divide entre 16?
2. ¿Para qué sirve `np.newaxis` aquí?
3. ¿Cuántas imágenes quedan para entrenar y para la prueba?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: imagenes / 16.0 y añade el eje del canal con [..., np.newaxis]
X = ...
X_ent, X_pru, y_ent, y_pru = ...

assert X.shape == (1797, 8, 8, 1), "X debe tener forma (1797, 8, 8, 1): ¿has añadido el canal?"
assert X.max() <= 1.0 and X.min() >= 0.0, "Normaliza dividiendo entre 16 para que quede entre 0 y 1"
assert len(X_ent) + len(X_pru) == 1797 and len(X_pru) == 450, "Reparto 75 % / 25 %"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Programar la convolución a mano

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué tamaño tiene la salida de convolucionar una imagen de 14×14 con un filtro 3×3 sin relleno?
2. ¿Qué se hace en cada posición del filtro?
3. ¿Qué diferencia hay entre este filtro y los de una CNN?

In [ ]:
def convolucion2d(imagen, filtro):
    """Convolución 'válida' (sin relleno, paso 1) de una imagen 2D con un filtro 2D."""
    alto, ancho = imagen.shape
    k_alto, k_ancho = filtro.shape
    salida = np.zeros((alto - k_alto + 1, ancho - k_ancho + 1))
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            zona = imagen[i:i + k_alto, j:j + k_ancho]
            salida[i, j] = np.sum(zona * filtro)
    return salida

### ✏️ A5 · EJERCICIO 2 de 4 — Calcular el tamaño de salida

**Tu turno:** Para una entrada de `n` píxeles, filtro de `k`, relleno (*padding*) `p` a cada lado y paso (*stride*) `s`, la salida mide `(n - k + 2p) / s + 1` (parte entera). Escribe `tam_salida(n, k, p=0, s=1)` que devuelva ese número entero. Sirve para entender por qué las imágenes "encogen" capa tras capa (y por qué Keras ofrece `padding="same"`).

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto da `tam_salida(8, 3)` y por qué?
2. ¿Qué tamaño tendría la salida de un filtro 5×5 sin relleno sobre una imagen de 32×32?
3. ¿Qué efecto tiene el relleno (`padding`)?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: aplica la fórmula con división entera (//)
def tam_salida(n, k, p=0, s=1):
    ...

assert tam_salida(8, 3) == 6, "8 píxeles con filtro 3 sin relleno dan 6"
assert tam_salida(8, 3, p=1) == 8, "Con relleno 1 el tamaño se conserva ('same')"
assert tam_salida(224, 7, p=3, s=2) == 112, "Caso típico de las primeras capas de ResNet: 224 → 112"
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Detectar bordes con un filtro (Sobel)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué bordes se iluminan con `filtro_vertical`?
2. ¿Cómo se obtiene el filtro horizontal a partir del vertical?
3. ¿Qué tamaño tiene la salida y por qué es menor que la imagen?

In [ ]:
# Imagen de prueba: rectángulo claro sobre fondo oscuro
prueba = np.zeros((14, 14)); prueba[3:11, 4:10] = 1.0
filtro_vertical = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])     # detecta bordes verticales (Sobel)
filtro_horizontal = filtro_vertical.T                                  # detecta bordes horizontales

fig, ejes = plt.subplots(1, 3, figsize=(11, 3.6))
for eje, im, titulo in zip(ejes, [prueba, convolucion2d(prueba, filtro_vertical), convolucion2d(prueba, filtro_horizontal)],
                           ["Imagen original", "Bordes verticales", "Bordes horizontales"]):
    eje.imshow(im, cmap="gray"); eje.set_title(titulo); eje.axis("off")
plt.tight_layout(); plt.show()
print("Tamaño de la salida:", convolucion2d(prueba, filtro_vertical).shape, "(14 - 3 + 1 = 12)")

### ✏️ A7 · EJERCICIO 3 de 4 — Programar el max pooling

**Tu turno:** Escribe `max_pooling(mapa, t=2)` que divida un mapa 2D (con lados múltiplos de `t`) en bloques de `t × t` y devuelva el máximo de cada bloque (salida de lado `alto // t`). Pista: recorre los bloques con dos bucles o busca cómo reorganizar el mapa con `reshape` para agrupar cada bloque.

**Debe salir:** ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué devuelve `max_pooling` con un mapa de 12×12 y bloques de 2×2?
2. ¿Qué ventaja tiene quedarse con el máximo de cada bloque?
3. En `[[1,2,0,0],[3,4,0,1],[0,0,5,6],[0,9,7,8]]`, ¿por qué sale `[[4,1],[9,8]]`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: dos bucles sobre los bloques (np.max de cada zona) o un reshape a 4 dimensiones
def max_pooling(mapa, t=2):
    ...

_m = np.array([[1, 2, 0, 0], [3, 4, 0, 1], [0, 0, 5, 6], [0, 9, 7, 8]])
assert np.array_equal(max_pooling(_m), [[4, 1], [9, 8]]), "Esperado [[4, 1], [9, 8]]: máximo de cada bloque 2×2"
assert max_pooling(np.zeros((12, 12))).shape == (6, 6), "Una imagen de 12×12 con bloques 2×2 debe dar 6×6"
print("✔ Ejercicio 3 correcto")

### 👀 A8 · EJEMPLO — Convolución + pooling sobre la imagen de prueba

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué tamaño tiene el mapa tras el pooling?
2. ¿Por qué se representa el valor absoluto?
3. ¿Qué se conserva y qué se pierde con el pooling?

In [ ]:
salida_conv = convolucion2d(prueba, filtro_vertical)
fig, ejes = plt.subplots(1, 2, figsize=(8, 3.6))
ejes[0].imshow(np.abs(salida_conv), cmap="gray"); ejes[0].set_title("Mapa 12×12 tras la convolución"); ejes[0].axis("off")
ejes[1].imshow(max_pooling(np.abs(salida_conv)), cmap="gray"); ejes[1].set_title("Tras max pooling 2×2 (6×6)"); ejes[1].axis("off")
plt.tight_layout(); plt.show()

### 👀 A9 · EJEMPLO — Entrenar un MLP de referencia

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Cuántos parámetros tiene el MLP y qué exactitud logra?
2. ¿Qué hace `Flatten`?
3. ¿Qué se pierde al aplanar la imagen?

In [ ]:
def construir_mlp_imagen():
    m = keras.Sequential([keras.layers.Input(shape=(8, 8, 1)), keras.layers.Flatten(),
                          keras.layers.Dense(64, activation="relu"), keras.layers.Dense(10, activation="softmax")])
    m.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

mlp = construir_mlp_imagen()
h_mlp = mlp.fit(X_ent, y_ent, validation_split=0.2, epochs=25, batch_size=32, verbose=0)
print("MLP · parámetros:", mlp.count_params(), "· exactitud en prueba:", round(mlp.evaluate(X_pru, y_pru, verbose=0)[1], 3))

### ✏️ A10 · EJERCICIO 4 de 4 — Construir la CNN

**Tu turno:** Escribe `construir_cnn(filtros=(16, 32))` que devuelva un modelo compilado con: `Input(8,8,1)` → por cada valor de `filtros`: `Conv2D(f, 3, padding="same", activation="relu")` + `MaxPooling2D(2)` → `Flatten` → `Dense(10, activation="softmax")`. Compílalo igual que el MLP (`adam`, `sparse_categorical_crossentropy`, métrica `accuracy`).

**Debe salir:** el resumen de la CNN (**6 090** parámetros) y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Cuántos parámetros tiene la CNN y cuántos tiene su primera capa convolucional?
2. ¿Cómo cambia el tamaño de los mapas a lo largo de la red?
3. ¿Por qué la capa de salida es `Dense(10, activation="softmax")`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: un bucle sobre filtros añadiendo Conv2D y MaxPooling2D; después Flatten y Dense(10, softmax)
def construir_cnn(filtros=(16, 32)):
    ...

cnn = construir_cnn()
cnn.summary()
assert sum(isinstance(c, keras.layers.Conv2D) for c in cnn.layers) == 2, "Deben ser 2 capas convolucionales"
assert sum(isinstance(c, keras.layers.MaxPooling2D) for c in cnn.layers) == 2, "Deben ser 2 capas de max pooling"
assert cnn.output_shape == (None, 10), "La salida debe tener 10 neuronas (una por dígito)"
assert cnn.count_params() == (3*3*1*16 + 16) + (3*3*16*32 + 32) + (2*2*32*10 + 10), "Parámetros inesperados: revisa filtros, padding='same' y el orden de las capas"
print("✔ Ejercicio 4 correcto")

### 👀 A11 · EJEMPLO — Entrenar la CNN y compararla con el MLP

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Qué exactitud tienen el MLP y la CNN en prueba?
2. ¿Por qué la ventaja de la CNN es pequeña en este conjunto?
3. ¿Por qué una CNN necesita muchos menos parámetros con fotos grandes?

In [ ]:
keras.utils.set_random_seed(SEMILLA)
cnn = construir_cnn()
h_cnn = cnn.fit(X_ent, y_ent, validation_split=0.2, epochs=30, batch_size=32, verbose=0)
acc_cnn = cnn.evaluate(X_pru, y_pru, verbose=0)[1]
acc_mlp = mlp.evaluate(X_pru, y_pru, verbose=0)[1]

plt.plot(h_cnn.history["val_accuracy"], label="CNN"); plt.plot(h_mlp.history["val_accuracy"], label="MLP")
plt.xlabel("época"); plt.ylabel("exactitud de validación"); plt.title("CNN frente a MLP en dígitos 8×8"); plt.legend(); plt.show()
print(f"MLP: {mlp.count_params()} parámetros, exactitud {acc_mlp:.3f}")
print(f"CNN: {cnn.count_params()} parámetros, exactitud {acc_cnn:.3f}")

### 👀 A12 · EJEMPLO — Filtros y mapas de activación

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué forma tienen los pesos de la primera capa y qué significa?
2. ¿Qué muestra un mapa de activación?
3. ¿Los filtros los eligió alguien?

In [ ]:
filtros_1 = cnn.layers[0].get_weights()[0]           # forma (3, 3, 1, 16)
fig, ejes = plt.subplots(1, 8, figsize=(12, 2))
for i, eje in enumerate(ejes):   # mostramos los 8 primeros filtros
    eje.imshow(filtros_1[:, :, 0, i], cmap="coolwarm"); eje.set_title(f"filtro {i}", fontsize=8); eje.axis("off")
plt.suptitle("Filtros 3×3 aprendidos por la primera capa (rojo = peso positivo, azul = negativo)"); plt.show()

extractor = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)
ejemplo = X_pru[:1]
mapas = extractor.predict(ejemplo, verbose=0)[0]     # (8, 8, 16)
fig, ejes = plt.subplots(1, 9, figsize=(13, 2))
ejes[0].imshow(ejemplo[0, :, :, 0], cmap="gray_r"); ejes[0].set_title(f"entrada ({y_pru[0]})", fontsize=8); ejes[0].axis("off")
for i in range(8):
    ejes[i + 1].imshow(mapas[:, :, i], cmap="viridis"); ejes[i + 1].set_title(f"mapa {i}", fontsize=8); ejes[i + 1].axis("off")
plt.show()

### 👀 A13 · EJEMPLO — Dígitos desplazados y capas de aumento

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A13).

1. ¿Qué hace `desplazar(X_pru, 1, 1)`?
2. ¿Por qué la fila inferior cambia cada vez que ejecutas la celda?
3. ¿Para qué sirve el aumento de datos?

In [ ]:
def desplazar(imagenes_, dx, dy):
    return np.roll(np.roll(imagenes_, dy, axis=1), dx, axis=2)

X_pru_desp = desplazar(X_pru, 1, 1)

aumento = keras.Sequential([keras.layers.RandomTranslation(0.12, 0.12, fill_mode="constant"),
                            keras.layers.RandomRotation(0.06, fill_mode="constant")])
lote = X_ent[:10]
fig, ejes = plt.subplots(2, 10, figsize=(13, 3))
for j in range(10):
    ejes[0, j].imshow(lote[j, :, :, 0], cmap="gray_r"); ejes[0, j].axis("off")
    ejes[1, j].imshow(aumento(lote[j:j + 1], training=True)[0, :, :, 0], cmap="gray_r"); ejes[1, j].axis("off")
ejes[0, 0].set_title("original", fontsize=8, loc="left"); ejes[1, 0].set_title("aumentada", fontsize=8, loc="left")
plt.show()

### ✏️ A14 · MINI-RETO — CNN con aumento de datos

**Tu turno:** Construye `cnn_aug`: una CNN igual que `construir_cnn()` pero con las capas de `aumento` **al principio** del modelo (usa `keras.Sequential([keras.layers.Input((8, 8, 1)), aumento, ...])` o añádelas con `add`). Entrénala 40 épocas (`epochs=40`, `batch_size=32`, `validation_split=0.2`) y guarda en `acc_aug_desp` su exactitud sobre `X_pru_desp`. Guarda también en `acc_cnn_desp` la de la CNN sin aumento. Responde con tus palabras (🧠).

**Debe salir:** `Con dígitos desplazados → CNN normal: 0.264 · CNN con aumento: 0.651 · MLP: 0.144` y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A14).

1. ¿Qué exactitud tienen la CNN normal, la CNN con aumento y el MLP con los dígitos desplazados?
2. ¿Qué conclusión sacas sobre el aumento de datos?
3. ¿Tiene el aumento algún coste?

In [ ]:
# pista: Sequential con Input, las capas de aumento, y después las Conv/Pool/Flatten/Dense; compila igual
keras.utils.set_random_seed(SEMILLA)
cnn_aug = ...
# entrénala 40 épocas con validation_split=0.2 y batch_size=32
acc_aug_desp = ...
acc_cnn_desp = ...

assert any(isinstance(c, keras.Sequential) or "random" in c.name for c in cnn_aug.layers), "Falta incluir las capas de aumento dentro del modelo"
assert 0.0 <= acc_aug_desp <= 1.0 and 0.0 <= acc_cnn_desp <= 1.0, "Las exactitudes deben estar entre 0 y 1"
print(f"Con dígitos desplazados → CNN normal: {acc_cnn_desp:.3f} · CNN con aumento: {acc_aug_desp:.3f} · MLP: {mlp.evaluate(X_pru_desp, y_pru, verbose=0)[1]:.3f}")
print("✔ Mini-reto correcto")

### 👀 A15 · EJEMPLO — Matriz de confusión de la CNN

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A15).

1. ¿Qué significan las filas y las columnas?
2. ¿Qué indican los números fuera de la diagonal?
3. ¿Cuántos errores comete la CNN y cuál es el más frecuente?

In [ ]:
pred = cnn.predict(X_pru, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_pru, pred)
plt.figure(figsize=(5.5, 5)); plt.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks(range(10)); plt.yticks(range(10)); plt.xlabel("predicho"); plt.ylabel("real"); plt.title("Matriz de confusión (CNN)"); plt.show()

### 👀 A16 · EJEMPLO — Rama Colab (opcional, requiere internet)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A16).

1. ¿Qué hace la celda con `USAR_INTERNET = False`?
2. ¿Para qué se usa `RandomFlip("horizontal")` en la rama de flores?

In [ ]:
USAR_INTERNET = False        # ← cámbialo a True solo si tienes conexión (Colab)

if USAR_INTERNET:
    ruta_flores = keras.utils.get_file(
        "flower_photos", untar=True,
        origin="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz")
    ruta_flores = os.path.join(os.path.dirname(ruta_flores), "flower_photos")
    ds_ent, ds_val = keras.utils.image_dataset_from_directory(
        ruta_flores, validation_split=0.2, subset="both", seed=SEMILLA, image_size=(96, 96), batch_size=32)
    print("Clases:", ds_ent.class_names)
    cnn_flores = keras.Sequential([
        keras.layers.Input((96, 96, 3)), keras.layers.Rescaling(1 / 255),
        keras.layers.RandomFlip("horizontal"),
        keras.layers.Conv2D(16, 3, activation="relu"), keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(32, 3, activation="relu"), keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation="relu"), keras.layers.MaxPooling2D(),
        keras.layers.Flatten(), keras.layers.Dropout(0.3), keras.layers.Dense(5, activation="softmax")])
    cnn_flores.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    cnn_flores.fit(ds_ent, validation_data=ds_val, epochs=8)
else:
    print("Rama de Colab desactivada (USAR_INTERNET = False). Sigue con el resto del notebook.")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `matplotlib`, `tensorflow`, `keras` y, de `scikit-learn`, el conjunto de dígitos, la separación de datos y las métricas.
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow.
    - `keras.utils.set_random_seed(SEMILLA)` fija las semillas para que los resultados se repitan.
- **Qué debe salir:**
    - `Listo ✔`. Importar TensorFlow tarda unos segundos.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.
    - `ModuleNotFoundError: No module named 'tensorflow'` → fuera de Colab, instala TensorFlow en tu entorno.

## 1. Una imagen es una matriz de números

#### A2 · Ejemplo — Cargar los dígitos y verlos como matrices

**Contexto TurisData.** El cliente tiene miles de fotos de alojamientos, zonas comunes y platos que hoy nadie usa. Antes de clasificarlas necesitas entender cómo una red **procesa una imagen**. Lo verás con imágenes pequeñas (dígitos 8×8 de `scikit-learn`) que se ejecutan sin internet.

**Idea.** Una imagen en escala de grises es una tabla de píxeles con valores (aquí de 0 = blanco, sin tinta, a 16 = negro, tinta máxima).
Una imagen en color tiene **3 canales** (rojo, verde, azul): una tabla por color. Su forma es
`(alto, ancho, canales)`. Un lote de imágenes añade una dimensión delante: `(n_imágenes, alto, ancho, canales)`.

- **Qué hace:**
    - `load_digits()` carga 1 797 imágenes de dígitos de 8×8 píxeles; `imagenes` son las matrices y `etiquetas` el dígito que representa cada una.
    - El primer `print` muestra la forma del conjunto y el rango de valores; el segundo, la matriz 8×8 de la primera imagen (`imagenes[0].astype(int)`).
    - El bucle dibuja con `imshow` las 16 primeras imágenes, cada una titulada con su etiqueta.
- **Qué debe salir:**
    - `Forma del conjunto: (1797, 8, 8) · valores de 0.0 a 16.0`, la matriz de la primera imagen (un 0, con valores de 0 a 15) y una rejilla de 16 dígitos en baja resolución.
- **Cómo leerlo:**
    - Para el ordenador, un dígito son **64 números** (de 0 = sin tinta a 16 = tinta máxima). Una foto en color tendría 3 canales: forma `(alto, ancho, 3)`.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene el conjunto y qué significa cada número?* → `(1797, 8, 8)`: **1 797 imágenes** de **8 × 8** píxeles.
2. *¿Qué valores puede tener un píxel y qué significan?* → De **0** (sin tinta, blanco) a **16** (tinta máxima, negro).
3. *¿Qué forma tendría un lote de fotos en color de 96×96?* → `(n_imágenes, 96, 96, 3)`: se añade una dimensión delante (el lote) y los **3 canales** (rojo, verde, azul) detrás.

</details>

#### A3 · Ejercicio 1 — Preparar los datos para una red

- **Qué hace:**
    - `(imagenes / 16.0).astype("float32")[..., np.newaxis]` divide entre 16 (así los valores quedan entre 0 y 1) y añade un último eje de tamaño 1: el **canal** (forma `(1797, 8, 8, 1)`).
    - `train_test_split(X, etiquetas, test_size=0.25, random_state=SEMILLA, stratify=etiquetas)` separa un 25 % para la prueba (450 imágenes) y un 75 % para entrenar (1 347), con la misma proporción de cada dígito.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `X debe tener forma (1797, 8, 8, 1): ¿has añadido el canal?` → falta `[..., np.newaxis]`.
    - `Normaliza dividiendo entre 16 ...` → los valores no están entre 0 y 1.
    - `Reparto 75 % / 25 %` → `test_size` no es 0.25 o no se han creado bien los cuatro conjuntos.
    - `NameError: name 'imagenes' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué se divide entre 16?* → Para **normalizar**: los píxeles pasan de 0–16 a **0–1**, una escala con la que las redes entrenan mejor.
2. *¿Para qué sirve `np.newaxis` aquí?* → Para añadir el **eje del canal** (`(8, 8)` → `(8, 8, 1)`): Keras espera una dimensión de canales, aunque sea una sola (escala de grises).
3. *¿Cuántas imágenes quedan para entrenar y para la prueba?* → **1 347** y **450**.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
X = (imagenes / 16.0).astype("float32")[..., np.newaxis]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, etiquetas, test_size=0.25, random_state=SEMILLA, stratify=etiquetas)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. La convolución: un filtro que se desliza

#### A4 · Ejemplo — Programar la convolución a mano

**Idea.** Un **filtro** (o *kernel*) es una mini-tabla, por ejemplo 3×3. Se coloca sobre una zona de la imagen,
se multiplica casilla a casilla, se suma y se obtiene **un** número. Después se desliza una posición y se repite.
El resultado es un **mapa de activación**: dice *dónde* aparece el patrón que busca el filtro (un borde vertical,
una esquina…). Una CNN **aprende sus propios filtros** durante el entrenamiento.

Vamos a hacerlo a mano con NumPy sobre una imagen sencilla: un rectángulo claro en fondo oscuro.

- **Qué hace:**
    - `convolucion2d(imagen, filtro)` calcula una convolución «válida» (sin relleno y con paso 1): crea una salida de tamaño `alto - k_alto + 1` por `ancho - k_ancho + 1`.
    - Dos bucles recorren cada posición; en cada una toma la zona de la imagen del tamaño del filtro, la multiplica casilla a casilla (`zona * filtro`) y suma todo (`np.sum`) para obtener **un** número.
- **Qué debe salir:**
    - No imprime nada: solo define la función.
- **Cómo leerlo:**
    - Es la operación que repite una capa convolucional miles de veces; la única diferencia es que la red **aprende** los números del filtro.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tamaño tiene la salida de convolucionar una imagen de 14×14 con un filtro 3×3 sin relleno?* → **12×12** (14 − 3 + 1).
2. *¿Qué se hace en cada posición del filtro?* → Se **multiplica** la zona por el filtro casilla a casilla y se **suma**: sale un solo número.
3. *¿Qué diferencia hay entre este filtro y los de una CNN?* → Aquí los números del filtro los fijamos nosotros; en una CNN la red **los aprende** durante el entrenamiento.

</details>

#### A5 · Ejercicio 2 — Calcular el tamaño de salida

- **Qué hace:**
    - `tam_salida(n, k, p=0, s=1)` aplica la fórmula `(n - k + 2p) // s + 1`: `n` es el tamaño de la entrada, `k` el del filtro, `p` el relleno a cada lado y `s` el paso.
    - La comprobación prueba tres casos: 8 con filtro 3 → 6; con relleno 1 → 8 (se conserva, como `padding="same"`); y 224 con filtro 7, relleno 3 y paso 2 → 112 (las primeras capas de ResNet).
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Sin relleno, la imagen **encoge** en cada capa: por eso Keras ofrece `padding="same"`.
- **Si falla:**
    - `8 píxeles con filtro 3 sin relleno dan 6` → la fórmula está mal (falta el `+ 1` o el paréntesis).
    - `Con relleno 1 el tamaño se conserva` → falta `2 * p` (el relleno se pone a cada lado).
    - `Caso típico de las primeras capas de ResNet ...` → falta dividir entre `s` (usa `//`).
    - `NameError: name 'tam_salida' is not defined` → no has ejecutado la celda del ejercicio.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto da `tam_salida(8, 3)` y por qué?* → **6**: (8 − 3 + 0) // 1 + 1 = 6.
2. *¿Qué tamaño tendría la salida de un filtro 5×5 sin relleno sobre una imagen de 32×32?* → **28** (32 − 5 + 1).
3. *¿Qué efecto tiene el relleno (`padding`)?* → Con el relleno adecuado (p = 1 para filtro 3) la salida **conserva el tamaño** de la entrada (`padding="same"`).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def tam_salida(n, k, p=0, s=1):
    return (n - k + 2 * p) // s + 1
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejemplo — Detectar bordes con un filtro (Sobel)

- **Qué hace:**
    - `prueba` es una imagen 14×14 con un rectángulo claro (valor 1) sobre fondo oscuro (valor 0).
    - `filtro_vertical` es el filtro de Sobel, que detecta bordes verticales; `filtro_horizontal` es su traspuesta (`.T`) y detecta bordes horizontales.
    - Se dibujan tres imágenes: la original y su convolución con cada filtro (`convolucion2d`).
- **Qué debe salir:**
    - Tres imágenes: el rectángulo, los bordes verticales (se iluminan sus lados izquierdo y derecho) y los horizontales (arriba y abajo), y `Tamaño de la salida: (12, 12) (14 - 3 + 1 = 12)`.
- **Cómo leerlo:**
    - Cada filtro responde donde aparece su patrón: el vertical se activa en los lados izquierdo y derecho del rectángulo (con valores de signo contrario en cada lado) y no en la parte de arriba y abajo. El **mapa de activación** dice *dónde* está el patrón.
- **Si falla:**
    - `NameError: name 'convolucion2d' is not defined` → falta el ejemplo de la convolución a mano.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué bordes se iluminan con `filtro_vertical`?* → Los **laterales** (izquierdo y derecho) del rectángulo.
2. *¿Cómo se obtiene el filtro horizontal a partir del vertical?* → **Trasponiéndolo** (`filtro_vertical.T`).
3. *¿Qué tamaño tiene la salida y por qué es menor que la imagen?* → **12×12**: la imagen es de 14×14 y el filtro 3×3 sin relleno hace que la salida «encoja» 2 píxeles por lado.

</details>

## 3. Pooling: quedarse con lo importante

#### A7 · Ejercicio 3 — Programar el max pooling

**Idea.** El *max pooling* divide el mapa en bloques (p. ej. 2×2) y se queda con el **máximo** de cada uno.
Resultado: mapas la mitad de grandes, menos cálculo y algo de **tolerancia a pequeños desplazamientos**
(si el borde se mueve un píxel dentro del bloque, el máximo no cambia).

- **Qué hace:**
    - `mapa.reshape(alto // t, t, ancho // t, t)` reorganiza el mapa en bloques de `t × t` (cuatro dimensiones) y `.max(axis=(1, 3))` se queda con el máximo de cada bloque.
    - La comprobación usa un mapa 4×4 de ejemplo (`_m`) y verifica que el resultado es `[[4, 1], [9, 8]]` y que uno de 12×12 da 6×6.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `Esperado [[4, 1], [9, 8]]` → no estás tomando el máximo de cada bloque 2×2 (o los bloques están mal formados).
    - `Una imagen de 12×12 con bloques 2×2 debe dar 6×6` → el resultado debe tener lado `alto // t`.
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué devuelve `max_pooling` con un mapa de 12×12 y bloques de 2×2?* → Un mapa de **6×6**: la mitad en cada lado.
2. *¿Qué ventaja tiene quedarse con el máximo de cada bloque?* → Menos cálculo y algo de **tolerancia a pequeños desplazamientos**: si el borde se mueve un píxel dentro del bloque, el máximo no cambia.
3. *En `[[1,2,0,0],[3,4,0,1],[0,0,5,6],[0,9,7,8]]`, ¿por qué sale `[[4,1],[9,8]]`?* → Es el **máximo de cada bloque** 2×2: 4 (arriba izquierda), 1, 9 y 8.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def max_pooling(mapa, t=2):
    alto, ancho = mapa.shape
    return mapa.reshape(alto // t, t, ancho // t, t).max(axis=(1, 3))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejemplo — Convolución + pooling sobre la imagen de prueba

- **Qué hace:**
    - `salida_conv` es la convolución vertical de la imagen de prueba (12×12).
    - Se dibuja su valor absoluto (`np.abs`, para ver el borde sea cual sea su signo) y el resultado de aplicarle tu `max_pooling` 2×2.
- **Qué debe salir:**
    - Dos imágenes: el mapa 12×12 tras la convolución y el mapa 6×6 tras el max pooling.
- **Cómo leerlo:**
    - El pooling conserva los bordes pero con **la mitad de tamaño**: menos cálculo con casi la misma información.
- **Si falla:**
    - `NameError: name 'max_pooling' is not defined` → falta el ejercicio 3 (o ejecuta su 🆘 solución).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tamaño tiene el mapa tras el pooling?* → **6×6** (el de la convolución es 12×12).
2. *¿Por qué se representa el valor absoluto?* → Porque los bordes tienen valores positivos y negativos y así **se ven los dos**.
3. *¿Qué se conserva y qué se pierde con el pooling?* → Se conservan las **activaciones más fuertes** y se pierde precisión sobre su posición exacta.

</details>

## 4. Una CNN completa en Keras

#### A9 · Ejemplo — Entrenar un MLP de referencia

**Arquitectura típica:** `[Conv → ReLU → Pooling] × varias veces` → **Flatten** (aplanar) → `Dense` → salida.
Las primeras capas detectan trazos simples; las siguientes los combinan en formas cada vez más complejas
(es la idea de LeNet, VGG o ResNet, solo que mucho más grandes).

Como referencia entrenaremos también un **MLP** que trata la imagen como una lista de 64 números y
**pierde la vecindad entre píxeles**.

- **Qué hace:**
    - `construir_mlp_imagen()` crea un MLP que trata la imagen como una lista de 64 números: `Flatten` → `Dense(64, relu)` → `Dense(10, softmax)`; se compila con `adam` y `sparse_categorical_crossentropy`.
    - `mlp.fit(...)` lo entrena 25 épocas (con el 20 % de entrenamiento como validación) y `evaluate` mide su exactitud con la prueba.
- **Qué debe salir:**
    - `MLP · parámetros: 4810 · exactitud en prueba: 0.964`.
- **Cómo leerlo:**
    - El MLP ya acierta el 96,4 % de los dígitos, pero al aplanar la imagen **pierde la vecindad entre píxeles**.
- **Si falla:**
    - `NameError: name 'X_ent' is not defined` → falta el ejercicio 1. Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos parámetros tiene el MLP y qué exactitud logra?* → **4 810** y **0,964**.
2. *¿Qué hace `Flatten`?* → Convierte la imagen `(8, 8, 1)` en una lista de **64** números.
3. *¿Qué se pierde al aplanar la imagen?* → La **vecindad entre píxeles**: la red no sabe qué píxeles estaban juntos.

</details>

#### A10 · Ejercicio 4 — Construir la CNN

- **Qué hace:**
    - `Sequential([Input(shape=(8, 8, 1))])` crea el modelo; el bucle añade, por cada valor de `filtros`, una `Conv2D(f, 3, padding="same", activation="relu")` y un `MaxPooling2D(2)`.
    - Después `Flatten` aplana el resultado y `Dense(10, activation="softmax")` da una probabilidad por dígito. Se compila igual que el MLP.
    - La comprobación construye la CNN, muestra su `summary()` y verifica el número de capas convolucionales y de pooling, la salida y el número de parámetros.
- **Qué debe salir:**
    - Una tabla `Model: "sequential_1"` con las capas: `conv2d` (salida 8×8×16, 160 parámetros), `max_pooling2d` (4×4×16), `conv2d_1` (4×4×32, 4 640), `max_pooling2d_1` (2×2×32), `flatten` (128) y `dense` (10, 1 290), con `Total params: 6,090`; y `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - Cada pooling reduce a la mitad el alto y el ancho (8 → 4 → 2) y cada convolución aumenta el número de mapas (1 → 16 → 32). Los parámetros de una convolución son (3×3×canales de entrada + 1) × filtros: (9 + 1)·16 = 160 y (9·16 + 1)·32 = 4 640.
- **Si falla:**
    - `Deben ser 2 capas convolucionales` / `... de max pooling` → el bucle no recorre `filtros`, o falta una de las dos capas dentro.
    - `La salida debe tener 10 neuronas` → la última capa no es `Dense(10, ...)`.
    - `Parámetros inesperados: revisa filtros, padding='same' y el orden de las capas` → falta `padding="same"` o el orden de las capas.
    - `AttributeError: 'NoneType' object has no attribute 'summary'` → falta `return m`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos parámetros tiene la CNN y cuántos tiene su primera capa convolucional?* → **6 090** en total; la primera capa tiene **160** ((3·3·1 + 1) · 16).
2. *¿Cómo cambia el tamaño de los mapas a lo largo de la red?* → 8×8 → (pooling) 4×4 → (pooling) 2×2: **se reduce a la mitad** tras cada pooling, y los mapas pasan de 16 a 32.
3. *¿Por qué la capa de salida es `Dense(10, activation="softmax")`?* → Porque hay **10 clases** (los dígitos) y `softmax` da una probabilidad por clase.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def construir_cnn(filtros=(16, 32)):
    m = keras.Sequential([keras.layers.Input(shape=(8, 8, 1))])
    for f in filtros:
        m.add(keras.layers.Conv2D(f, 3, padding="same", activation="relu"))
        m.add(keras.layers.MaxPooling2D(2))
    m.add(keras.layers.Flatten())
    m.add(keras.layers.Dense(10, activation="softmax"))
    m.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A11 · Ejemplo — Entrenar la CNN y compararla con el MLP

Comparamos las dos redes con las mismas imágenes de prueba.

- **Qué hace:**
    - `cnn.fit(...)` entrena la CNN 30 épocas y se calculan las exactitudes en prueba de la CNN y del MLP.
    - El gráfico compara la exactitud de validación por época y los `print` muestran los parámetros y la exactitud de cada modelo.
- **Qué debe salir:**
    - Un gráfico con dos curvas y `MLP: 4810 parámetros, exactitud 0.964` y `CNN: 6090 parámetros, exactitud 0.976`.
- **Cómo leerlo:**
    - **Qué observar.** En imágenes minúsculas y limpias un MLP ya rinde muy bien: 0,964 frente a 0,976. La CNN gana poco aquí, pero su ventaja crece con imágenes más grandes: comparte los mismos filtros por toda la imagen y respeta la vecindad.
    - **Parámetros.** Aquí (8×8) el MLP no es más grande que la CNN; pero con fotos de 96×96×3 (27 648 valores de entrada) un MLP con 64 neuronas en la primera capa ya necesitaría ≈ 1,8 millones de pesos, mientras que una capa convolucional de 3×3 con 16 filtros usa 448, sea cual sea el tamaño de la foto.
- **Si falla:**
    - `NameError: name 'construir_cnn' is not defined` → falta el ejercicio 4 (o su 🆘 solución). Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud tienen el MLP y la CNN en prueba?* → **0,964** y **0,976**.
2. *¿Por qué la ventaja de la CNN es pequeña en este conjunto?* → Porque las imágenes son **minúsculas y limpias** (8×8): un MLP ya las resuelve bien.
3. *¿Por qué una CNN necesita muchos menos parámetros con fotos grandes?* → Porque **comparte los filtros por toda la imagen**: sus parámetros no dependen del tamaño de la foto (448 con 16 filtros 3×3 sobre 3 canales), mientras que un MLP conecta cada píxel con cada neurona.

</details>

## 5. Mirar dentro de la red

#### A12 · Ejemplo — Filtros y mapas de activación

Los **filtros** de la primera capa son pequeñas plantillas de trazos; los **mapas de activación** muestran dónde
se activa cada filtro con un dígito concreto.

- **Qué hace:**
    - `cnn.layers[0].get_weights()[0]` son los pesos de la primera capa: forma `(3, 3, 1, 16)`, es decir, 16 filtros de 3×3. Se dibujan los 8 primeros (rojo = peso positivo, azul = negativo).
    - `extractor` es un modelo auxiliar que devuelve la salida de la primera capa; con él se obtienen los 16 mapas de activación (`(8, 8, 16)`) de la primera imagen de prueba.
    - Se dibuja la imagen de entrada junto a sus 8 primeros mapas.
- **Qué debe salir:**
    - Una fila con 8 filtros de 3×3 y otra con la imagen de entrada seguida de 8 mapas de activación.
- **Cómo leerlo:**
    - Los filtros que ha aprendido la red son pequeñas **plantillas de trazos**; cada mapa muestra **dónde** se activa su filtro sobre ese dígito.
- **Si falla:**
    - `NameError: name 'cnn' is not defined` → falta entrenar la CNN.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tienen los pesos de la primera capa y qué significa?* → `(3, 3, 1, 16)`: **16 filtros** de **3×3** sobre **1** canal.
2. *¿Qué muestra un mapa de activación?* → **Dónde** se activa un filtro concreto sobre una imagen concreta.
3. *¿Los filtros los eligió alguien?* → **No**: la red los aprendió durante el entrenamiento.

</details>

## 6. Data augmentation

#### A13 · Ejemplo — Dígitos desplazados y capas de aumento

**Idea.** Si el alojamiento aparece más a la izquierda, más pequeño o ligeramente girado, sigue siendo el mismo
alojamiento. El *data augmentation* genera **variaciones aleatorias** del entrenamiento (solo al entrenar) para que
la red no dependa de la posición exacta. Keras lo hace con capas (`RandomTranslation`, `RandomRotation`, `RandomZoom`,
`RandomFlip`) que se colocan al principio del modelo.

Prueba de estrés: evaluaremos ambos modelos con **dígitos desplazados un píxel** (a la derecha y hacia abajo; una situación
realista que el entrenamiento no vio).

- **Qué hace:**
    - `desplazar(imagenes_, dx, dy)` mueve las imágenes `dx` píxeles a la derecha y `dy` hacia abajo (con `np.roll`); `X_pru_desp` es la prueba desplazada 1 píxel en ambas direcciones.
    - `aumento` es un modelo con dos capas aleatorias: `RandomTranslation` (hasta un 12 % del tamaño, aproximadamente 1 píxel) y `RandomRotation(0.06)` (unos ±22°).
    - Se dibujan 10 dígitos originales (fila superior) y su versión aumentada (inferior; `training=True` activa el azar).
- **Qué debe salir:**
    - Dos filas de 10 dígitos: la superior original y la inferior con pequeños desplazamientos y giros.
- **Cómo leerlo:**
    - Cada vez que se ejecuta, la fila inferior cambia: **son variaciones nuevas** de la misma imagen. Esta aleatoriedad solo se aplica al entrenar.
- **Si falla:**
    - `NameError: name 'X_ent' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué hace `desplazar(X_pru, 1, 1)`?* → Mueve **1 píxel a la derecha y 1 hacia abajo** cada imagen de prueba.
2. *¿Por qué la fila inferior cambia cada vez que ejecutas la celda?* → Porque las capas de aumento aplican **desplazamientos y giros aleatorios**.
3. *¿Para qué sirve el aumento de datos?* → Para que la red **no dependa de la posición exacta**: ve variaciones realistas durante el entrenamiento.

</details>

#### A14 · Mini-reto — CNN con aumento de datos

- **Qué hace:**
    - `keras.Sequential([Input(shape=(8, 8, 1)), aumento])` empieza el modelo con las capas de aumento; después se añaden las mismas capas `Conv2D`/`MaxPooling2D`/`Flatten`/`Dense` que en `construir_cnn`, y se compila igual.
    - `cnn_aug.fit(..., epochs=40, batch_size=32, validation_split=0.2)` la entrena; el aumento solo actúa durante el entrenamiento.
    - `acc_aug_desp` y `acc_cnn_desp` son las exactitudes de la CNN con aumento y sin aumento sobre las imágenes desplazadas (`X_pru_desp`).
- **Qué debe salir:**
    - `Con dígitos desplazados → CNN normal: 0.264 · CNN con aumento: 0.651 · MLP: 0.144` y `✔ Mini-reto correcto`.
- **Cómo leerlo:**
    - Con un desplazamiento de 1 píxel, la CNN normal cae de 0,976 a 0,264 y el MLP a 0,144. El aumento sube la CNN a 0,651: **mejora mucho**, aunque no la deja como antes. Además, el aumento tiene un coste: en la prueba **sin desplazar** esa CNN acierta 0,898 (frente a 0,976).
- **Si falla:**
    - `Falta incluir las capas de aumento dentro del modelo` → `aumento` no está dentro de `cnn_aug`.
    - `Las exactitudes deben estar entre 0 y 1` → has guardado otra cosa (por ejemplo la pérdida) en lugar del valor de `evaluate(...)[1]`.
    - `NameError: name 'aumento' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud tienen la CNN normal, la CNN con aumento y el MLP con los dígitos desplazados?* → **0,264**, **0,651** y **0,144**.
2. *¿Qué conclusión sacas sobre el aumento de datos?* → Que **hace a la red mucho más robusta** al desplazamiento (de 0,264 a 0,651), aunque no del todo, y que el MLP lo sufre más (0,144).
3. *¿Tiene el aumento algún coste?* → Sí: con imágenes **sin desplazar**, la CNN con aumento acierta 0,898 frente a 0,976 de la normal: la tarea de aprender es más difícil.

</details>

#### A15 · Ejemplo — Matriz de confusión de la CNN

### Matriz de confusión de la CNN: dónde se equivoca

- **Qué hace:**
    - `cnn.predict(X_pru).argmax(axis=1)` da el dígito más probable de cada imagen de prueba (`pred`).
    - `confusion_matrix(y_pru, pred)` cuenta, para cada dígito real (filas), cuántas veces se predijo cada dígito (columnas); se dibuja con números en cada casilla no vacía.
- **Qué debe salir:**
    - Una matriz 10×10 casi diagonal con unos pocos números fuera de ella.
- **Cómo leerlo:**
    - Solo hay **11 errores** en 450 imágenes. Los más frecuentes: un 5 que se predice como 9 (3 veces), un 8 como 1 (2) y un 1 como 9 (2).
- **Si falla:**
    - `NameError: name 'cnn' is not defined` → falta entrenar la CNN.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué significan las filas y las columnas?* → **Filas = dígito real**, **columnas = dígito predicho**.
2. *¿Qué indican los números fuera de la diagonal?* → **Errores**: la red predijo un dígito distinto del real.
3. *¿Cuántos errores comete la CNN y cuál es el más frecuente?* → **11** de 450; el más frecuente, un **5** predicho como **9** (3 veces).

</details>

## 7. Rama Colab (opcional): fotos reales

#### A16 · Ejemplo — Rama Colab (opcional, requiere internet)

> **AVISO: requiere internet. No se ejecuta en el taller sin conexión.** Está pensada para Colab en casa.
> Descarga ≈ 220 MB de fotos de 5 tipos de flores. Cambia `USAR_INTERNET` a `True` para probarla.
> **Esta celda no se ha verificado en el entorno del taller (sin internet).**

- **Qué hace:**
    - Con `USAR_INTERNET = False` (el valor por defecto) la celda solo escribe un mensaje. Si lo cambias a `True`, descarga las fotos de flores, las lee con `image_dataset_from_directory` y entrena una CNN pequeña con `RandomFlip`, tres bloques `Conv2D`/`MaxPooling2D` y `Dropout`.
    - Es opcional: no se ejecuta sin conexión y no hace falta para el resto del cuaderno.
- **Qué debe salir:**
    - Con el valor por defecto: `Rama de Colab desactivada (USAR_INTERNET = False). Sigue con el resto del notebook.`
- **Cómo leerlo:**
    - Esta rama **no se ha verificado** sin internet. Para probarla, en Colab, cambia `USAR_INTERNET` a `True` (descarga unos 220 MB).
- **Si falla:**
    - Con `USAR_INTERNET = True` sin conexión → error de descarga: déjalo en `False`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué hace la celda con `USAR_INTERNET = False`?* → Solo escribe el mensaje de que la rama está desactivada.
2. *¿Para qué se usa `RandomFlip("horizontal")` en la rama de flores?* → Para **aumentar los datos**: una flor volteada sigue siendo la misma flor.

</details>

## 8. Preguntas de reflexión
1. ¿Por qué, en imágenes grandes, una CNN necesita muchos menos parámetros que un MLP?
2. Si aplicamos un filtro 5×5 sin relleno a una imagen de 32×32, ¿qué tamaño tiene la salida? ¿Y con `padding="same"`?
3. ¿En qué casos sería un error aplicar `RandomFlip("horizontal")` a las fotos de TurisData? (Piensa en carteles con texto.)
4. ¿Por qué el *data augmentation* no se aplica al conjunto de prueba?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque en una capa convolucional **todos los píxeles comparten los mismos filtros** pequeños: sus parámetros dependen del tamaño del filtro y de su número, no del tamaño de la imagen. Un MLP conecta cada píxel con cada neurona (27 648 entradas × 64 neuronas ≈ 1,8 millones frente a 448 de una capa convolucional 3×3 con 16 filtros).
2. Sin relleno: (32 − 5)/1 + 1 = **28×28**. Con `padding="same"`: **32×32**.
3. Cuando el **sentido depende de la orientación**: un cartel con texto volteado se vuelve ilegible y una señal o un logotipo pierde su significado. En fotos de habitaciones o de paisajes no hay problema.
4. Porque la prueba debe medir **cómo se comporta con imágenes reales tal como llegarán**; el aumento es una ayuda para *enseñar* a la red durante el entrenamiento, y alterar la prueba falsearía la medida.

</details>

## 9. Resumen: qué has aprendido
- Una imagen es un tensor `(alto, ancho, canales)`; se normaliza antes de entrenar.
- Convolución = filtro deslizante que detecta patrones; pooling = reducir quedándose con lo más fuerte.
- Arquitectura típica: `[Conv → ReLU → Pool] × n → Flatten → Dense`.
- Una CNN comparte filtros por toda la imagen: menos parámetros; con *pooling* y aumento de datos tolera mejor los desplazamientos (sin aumento, un desplazamiento de 1 píxel ya la hunde).
- *Data augmentation* = variaciones aleatorias solo en entrenamiento para generalizar mejor.